## 🎯 Learning Objectives
* Understand the fundamental concepts of asynchronous task execution and parallelism in the context of AI agents.
* Identify scenarios where asynchronous execution significantly improves the performance and responsiveness of CrewAI agents.
* Implement and observe asynchronous task execution within a CrewAI framework using custom asynchronous tools.
* Analyze the performance implications and trade-offs associated with parallel processing in agentic workflows.


## Asynchronous Task Execution and Parallelism in CrewAI

In the rapidly evolving landscape of AI agents, efficiency and responsiveness are paramount. As agents become more sophisticated, they often need to perform multiple long-running operations, such as making API calls, querying databases, or interacting with large language models (LLMs). If these operations are executed sequentially, the agent's overall performance can suffer dramatically, leading to slow response times and underutilized resources.

This is where **asynchronous task execution** and **parallelism** become critical. Instead of waiting for one task to complete before starting the next, asynchronous execution allows an agent to initiate a task and then move on to other work while the first task runs in the background. When the first task finishes, its result can be processed. Parallelism takes this a step further, enabling multiple tasks to truly run *at the same time*, leveraging multi-core processors or concurrent I/O operations.

### The Restaurant Analogy

Imagine a busy restaurant kitchen:

*   **Synchronous Execution (One Chef):** A single chef takes an order, cooks the appetizer, waits for it to finish, then cooks the main course, waits for it to finish, and finally prepares dessert. The entire meal takes a long time because the chef can only do one thing at a time.

*   **Asynchronous Execution (One Chef, Smart Workflow):** The same chef takes an order. They put the appetizer in the oven (which takes time), then immediately start chopping vegetables for the main course. While the vegetables are cooking, they might start preparing the dessert. They constantly check on the oven and stove, attending to tasks as they complete or require attention. The chef is still one person, but they're not *waiting idly*.

*   **Parallel Execution (Multiple Chefs):** Now, imagine multiple chefs. Chef A cooks the appetizer, Chef B cooks the main course, and Chef C prepares dessert, all simultaneously. The entire meal is ready much faster because different parts are handled concurrently.

### Why is this crucial for AI Agents?

AI agents, especially those built with frameworks like CrewAI, frequently interact with external services. Each LLM call, API request, or tool execution can introduce latency. By employing asynchronous and parallel patterns, CrewAI agents can:

1.  **Reduce Latency:** Agents can perform multiple LLM calls or tool executions concurrently, drastically cutting down the total time required for complex workflows.
2.  **Improve Responsiveness:** Users receive results faster, leading to a more fluid and engaging experience.
3.  **Optimize Resource Utilization:** While one task is waiting for an I/O operation (like an API response), the agent can use its CPU cycles to process another task, making better use of available computing resources.
4.  **Enable Complex Workflows:** Many real-world problems require agents to gather information from various sources simultaneously or perform parallel sub-tasks before synthesizing a final answer.

CrewAI, built on Python's `asyncio` capabilities, inherently supports asynchronous operations. When you define tasks that involve LLM calls or custom tools that are designed to be `async`, CrewAI's execution engine can intelligently schedule and run these tasks concurrently, transforming your agent from a sequential bottleneck into a highly efficient, parallel processing powerhouse. This lesson will demonstrate how to leverage these capabilities to build faster, more robust AI agents.


In [ ]:
import os
import asyncio
from crewai import Agent, Task, Crew, Process
from crewai_tools import BaseTool
from typing import Type, Optional
from pydantic import BaseModel, Field

# --- 1. Set up Environment and Mock LLM (for demonstration) ---
# In a real scenario, you'd configure your actual LLM here.
# For this example, we'll use a mock to control response times.
class MockLLM:
    async def chat(self, messages, **kwargs):
        # Simulate LLM processing time
        await asyncio.sleep(1.5) 
        return {"content": "Mock LLM response for: " + messages[-1]['content']}

    def invoke(self, prompt, **kwargs):
        # Synchronous invoke for non-async contexts if needed, though we focus on async
        return {"content": "Mock LLM response for: " + prompt}

    async def ainvoke(self, prompt, **kwargs):
        # Asynchronous invoke
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response for: " + prompt}

    # CrewAI expects certain methods, even if not fully implemented for mock
    def __call__(self, *args, **kwargs):
        return self.invoke(*args, **kwargs)

    async def agenerate(self, messages, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response for: " + messages[0].content}

    async def acreate(self, messages, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response for: " + messages[0].content}

    async def acompletion(self, prompt, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response for: " + prompt}

    async def achat_completion(self, messages, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response for: " + messages[0].content}

    async def aget_chat_completion(self, messages, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response for: " + messages[0].content}

    async def aget_completion(self, prompt, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response for: " + prompt}

    async def _acall(self, *args, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response"}

    async def _agenerate(self, *args, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response"}

    async def _acreate(self, *args, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response"}

    async def _acompletion(self, *args, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response"}

    async def _achat_completion(self, *args, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response"}

    async def _aget_chat_completion(self, *args, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response"}

    async def _aget_completion(self, *args, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response"}

    async def _acall_llm(self, *args, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response"}

    async def _agenerate_llm(self, *args, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response"}

    async def _acreate_llm(self, *args, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response"}

    async def _acompletion_llm(self, *args, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response"}

    async def _achat_completion_llm(self, *args, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response"}

    async def _aget_chat_completion_llm(self, *args, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response"}

    async def _aget_completion_llm(self, *args, **kwargs):
        await asyncio.sleep(1.5)
        return {"content": "Mock LLM response"}


mock_llm = MockLLM()

# --- 2. Define Asynchronous Tools ---
# These tools simulate long-running I/O operations (e.g., API calls, database queries)

class MarketResearchInput(BaseModel):
    topic: str = Field(description="The market topic to research.")

class MarketResearchTool(BaseTool):
    name: str = "Market Research Tool"
    description: str = (
        "Gathers comprehensive market data and trends for a given topic. "
        "Simulates a network call that takes 3 seconds."
    )
    args_schema: Type[BaseModel] = MarketResearchInput

    async def _arun(self, topic: str) -> str:
        print(f"[MarketResearchTool] Starting research for '{topic}'...")
        await asyncio.sleep(3) # Simulate a long-running API call
        print(f"[MarketResearchTool] Finished research for '{topic}'.")
        return f"Detailed market report on {topic}: Key trends, competitor analysis, and growth projections."

    def _run(self, topic: str) -> str:
        # Synchronous fallback, though we'll focus on async
        print(f"[MarketResearchTool] Starting (sync) research for '{topic}'...")
        import time
        time.sleep(3)
        print(f"[MarketResearchTool] Finished (sync) research for '{topic}'.")
        return f"Detailed market report on {topic}: Key trends, competitor analysis, and growth projections."

class CompetitorAnalysisInput(BaseModel):
    company_name: str = Field(description="The name of the competitor company to analyze.")

class CompetitorAnalysisTool(BaseTool):
    name: str = "Competitor Analysis Tool"
    description: str = (
        "Analyzes a competitor's strategy, products, and market position. "
        "Simulates a network call that takes 2 seconds."
    )
    args_schema: Type[BaseModel] = CompetitorAnalysisInput

    async def _arun(self, company_name: str) -> str:
        print(f"[CompetitorAnalysisTool] Starting analysis for '{company_name}'...")
        await asyncio.sleep(2) # Simulate another long-running API call
        print(f"[CompetitorAnalysisTool] Finished analysis for '{company_name}'.")
        return f"Comprehensive analysis of {company_name}: Strengths, weaknesses, and market impact."

    def _run(self, company_name: str) -> str:
        # Synchronous fallback
        print(f"[CompetitorAnalysisTool] Starting (sync) analysis for '{company_name}'...")
        import time
        time.sleep(2)
        print(f"[CompetitorAnalysisTool] Finished (sync) analysis for '{company_name}'.")
        return f"Comprehensive analysis of {company_name}: Strengths, weaknesses, and market impact."

# Instantiate our async tools
market_research_tool = MarketResearchTool()
competitor_analysis_tool = CompetitorAnalysisTool()

# --- 3. Define Agents ---

researcher = Agent(
    role='Senior Market Researcher',
    goal='Gather and synthesize comprehensive market data and trends.',
    backstory='An expert in market dynamics, capable of identifying key opportunities and threats.',
    tools=[market_research_tool],
    llm=mock_llm,
    verbose=True
)

analyst = Agent(
    role='Competitive Strategy Analyst',
    goal='Analyze competitor strategies and provide actionable insights.',
    backstory='A seasoned analyst with a keen eye for competitive advantages and market positioning.',
    tools=[competitor_analysis_tool],
    llm=mock_llm,
    verbose=True
)

# --- 4. Define Tasks ---

# Task 1: Market Research (uses MarketResearchTool, takes 3 seconds)
market_research_task = Task(
    description=(
        "Conduct a detailed market research on 'AI-powered automation tools'. "
        "Identify key trends, emerging technologies, and potential market size. "
        "Use the 'Market Research Tool' to gather data."
    ),
    agent=researcher,
    expected_output="A detailed report on AI-powered automation tools market, including trends and size."
)

# Task 2: Competitor Analysis (uses CompetitorAnalysisTool, takes 2 seconds)
competitor_analysis_task = Task(
    description=(
        "Analyze 'AgenticLabs.ng' as a key competitor in the AI automation space. "
        "Focus on their product offerings, market strategy, and unique selling propositions. "
        "Use the 'Competitor Analysis Tool' to gather information."
    ),
    agent=analyst,
    expected_output="A comprehensive analysis of AgenticLabs.ng's competitive strategy."
)

# --- 5. Create and Run the Crew ---

# We'll run this in a synchronous context for demonstration, 
# but CrewAI's internal execution will leverage async for tools.

print("\n--- Starting Crew with Asynchronous Tasks ---")

# Create a Crew with both tasks. CrewAI will manage their execution.
# By default, CrewAI will try to run independent tasks concurrently if their underlying tools/LLM calls are async.
project_crew = Crew(
    agents=[researcher, analyst],
    tasks=[market_research_task, competitor_analysis_task],
    process=Process.sequential, # Even with sequential process, internal tool calls can be async
    manager_llm=mock_llm, # Manager also needs an LLM
    verbose=True
)

# Kickoff the crew. The `kickoff()` method is designed to handle async operations.
# We'll measure the total time taken.
import time
start_time = time.time()

# The kickoff method itself is synchronous, but it orchestrates async tasks internally.
result = project_crew.kickoff()

end_time = time.time()
total_time = end_time - start_time

print("\n--- Crew Execution Finished ---")
print(f"Total execution time: {total_time:.2f} seconds")
print("\nFinal Report:")
print(result)

# --- Expected Behavior ---
# If tasks were purely sequential (e.g., synchronous tools), total time would be ~3s (research) + ~2s (analysis) + LLM overhead = ~5s+.
# With asynchronous tools, the 3s and 2s operations can overlap. The total time should be closer to the longest individual task (3s) + LLM overhead.
# The mock LLM also adds 1.5s per call. So, 3s (tool) + 1.5s (LLM for tool usage) + 1.5s (LLM for final synthesis) = ~6s.
# If the LLM calls for each agent's thought process are also async and overlap, it could be even faster.
# The key is observing the `[MarketResearchTool] Starting...` and `[CompetitorAnalysisTool] Starting...` messages appearing close together, 
# and the total time being less than the sum of individual tool execution times.


### Interpreting the Output and Performance Trade-offs

When you run the code above, pay close attention to the timestamps (or the order of `print` statements if timestamps aren't explicit) from the `MarketResearchTool` and `CompetitorAnalysisTool`. You should observe that:

1.  The `[MarketResearchTool] Starting...` and `[CompetitorAnalysisTool] Starting...` messages appear relatively close to each other, indicating that both tools began their simulated long-running operations almost concurrently.
2.  The `[CompetitorAnalysisTool] Finished...` message appears before `[MarketResearchTool] Finished...`, as it has a shorter simulated delay (2 seconds vs. 3 seconds).
3.  The `Total execution time` reported at the end should be significantly less than the sum of the individual tool delays (3 seconds + 2 seconds = 5 seconds). Instead, it should be closer to the duration of the *longest* individual task (3 seconds) plus any overhead from LLM calls and CrewAI's orchestration. In our example, with mock LLM calls adding 1.5s each, a total time around 4.5-6 seconds is expected, demonstrating that the 2-second task ran *in parallel* with the 3-second task, rather than waiting for it.

This demonstrates the power of asynchronous execution: while the `MarketResearchTool` was busy simulating its 3-second API call, the `CompetitorAnalysisTool` was able to start and even finish its 2-second operation. CrewAI, by leveraging Python's `asyncio`, manages this concurrency seamlessly when your tools and LLM integrations are designed to be asynchronous.

### Performance Trade-offs and Use Cases

**When Parallelism Helps (and when it doesn't):**

*   **I/O-Bound Tasks (Helps Greatly):** Tasks that spend most of their time waiting for external resources (network requests, database queries, file I/O, LLM API calls) are prime candidates for asynchronous execution. While one task waits, the CPU can switch to another, maximizing throughput.
*   **CPU-Bound Tasks (Limited Help):** Tasks that heavily utilize the CPU (e.g., complex mathematical computations, heavy data processing) won't see significant gains from `asyncio` alone in Python due to the Global Interpreter Lock (GIL). For true parallelism in CPU-bound scenarios, you'd typically need multi-processing (using `multiprocessing` module) or distributed computing, which CrewAI can integrate with but doesn't directly provide through `asyncio`.
*   **Overhead:** Managing asynchronous tasks introduces some overhead. For very short, simple tasks, the overhead might outweigh the benefits. However, for typical AI agent workflows involving LLM calls and external tools, the benefits almost always far outweigh the overhead.

**Typical Use Cases in Agentic AI:**

*   **Concurrent Research:** An agent needs to gather information from multiple web sources or databases simultaneously.
*   **Multi-API Integration:** An agent interacts with several different APIs (e.g., weather API, stock market API, news API) to gather diverse data points for a single query.
*   **Parallel Sub-tasks:** A complex task can be broken down into independent sub-tasks that can be executed by different agents or tools concurrently.
*   **Real-time Monitoring:** Agents continuously monitor multiple data streams or events in parallel.
*   **Batch Processing:** Processing a large batch of inputs where each input can be handled independently by an agent or tool.

By understanding and implementing asynchronous patterns, you can build CrewAI agents that are not only intelligent but also incredibly fast and efficient, capable of handling complex, real-world business automation challenges.


### Resources

*   **CrewAI Documentation - Asynchronous Execution:** [https://docs.crewai.com/how-to/Async-Tasks/](https://docs.crewai.com/how-to/Async-Tasks/)
*   **Python `asyncio` Official Documentation:** [https://docs.python.org/3/library/asyncio.html](https://docs.python.org/3/library/asyncio.html)
*   **CrewAI Tools Documentation (for custom async tools):** [https://docs.crewai.com/how-to/Custom-Tools/](https://docs.crewai.com/how-to/Custom-Tools/)
*   **Real Python - Async IO in Python:** [https://realpython.com/async-io-python/](https://realpython.com/async-io-python/)
*   **Google AI Studio (for advanced LLM integration):** [https://aistudio.google.com/](https://aistudio.google.com/)
*   **Hugging Face (for open-source models and tools):** [https://huggingface.co/](https://huggingface.co/)
